In [6]:
%%script False

#import bacdive

#client = bacdive.BacdiveClient()

import requests

base_url = "https://api.bacdive.dsmz.de/v2"

genus = 'Lactiplantibacillus'
species = 'plantarum'

results = requests.get(f"{base_url}/taxon/{genus}/{species}")

strain_id = results.json().get('results')[0]

info = requests.get(f"{base_url}/fetch/{strain_id}")

print(info.json())

Couldn't find program: 'False'


In [7]:
import pandas
import pubchempy as pcp
import math
import time
import json

#so i dont keep having to edit lines of code
methods = {
    'utilisation': 'Utilized metabolite',
    'production': 'Produced metabolite'
}

method = 'production'

metabolites_csv = pandas.read_csv(f'..\\..\\dissertation DLC content\\metabolite_{method}.csv', skiprows=3, header=0)

In [8]:
microbe_metabolites_map = {}

pcp_chemical_mapping = {}

saved_microbe = metabolites_csv['species'][0]

metabolites_considered = []

for i in range(len(metabolites_csv)):
    microbe = metabolites_csv['species'][i]
    if str(microbe) == 'nan':
        microbe = saved_microbe
    else:
        if microbe != saved_microbe:
            saved_microbe = microbe

    if not microbe_metabolites_map.get(microbe):
        microbe_metabolites_map[microbe] = []

    metabolite = metabolites_csv[methods[method]][i]

    #if the synonyms have not already been added to the current microbe being looked at
    if metabolite in microbe_metabolites_map[microbe]:
        continue
    else:
        microbe_metabolites_map[microbe].append(metabolite)

    if not metabolite in metabolites_considered:
        #attempt to retrieve the synonyms
        try:
            pcp_resp = pcp.get_compounds(metabolite, 'name')
            time.sleep(0.2) #no more than 5 per second
            #if there is a response
            if pcp_resp:
                new_metabolites = [met.lower() for met in pcp_resp[0].synonyms] + [metabolite]
                #add the synonyms to the mapping
                pcp_chemical_mapping[pcp_resp[0].synonyms[0].lower()] = new_metabolites
            else:
                #otherwise just make the mapping the name of the csv metabolite itself
                pcp_chemical_mapping[metabolite] = [metabolite]
        #if the web retrieval fails for some reason
        except:
            #do the same thing here
            pcp_chemical_mapping[metabolite] = [metabolite]
        metabolites_considered.append(metabolite)

with open(f'..\\list_files\\metabolite_{method}.json', 'w', encoding='utf-8') as f:
    json.dump(microbe_metabolites_map, f)

with open(f'..\\list_files\\pcp_metabolite_{method}_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(pcp_chemical_mapping, f)

In [9]:
%%script False

new_pcp_mapping = {}

for chem, synonyms in pcp_chemical_mapping.items():
    new_pcp_mapping[synonyms[0]] = pcp_chemical_mapping[chem]

with open(f'..\\list_files\\pcp_metabolite_{method}_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(new_pcp_mapping, f)

Couldn't find program: 'False'


In [10]:
%%script False

method = 'utilisation'

with open(f'..\\list_files\\pcp_metabolite_{method}_mapping.json', 'r', encoding='utf-8') as f:
    old_pcp_mapping = json.load(f)

new_pcp_mapping = {}

for chem, synonyms in old_pcp_mapping.items():
    new_pcp_mapping[synonyms[0]] = old_pcp_mapping[chem]

with open(f'..\\list_files\\pcp_metabolite_{method}_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(new_pcp_mapping, f)

Couldn't find program: 'False'
